# Exercise 05 — Pandas Basics

You'll use Pandas when preparing data for RAG pipelines: loading documents from CSVs, filtering by metadata, grouping by source, exporting chunks. You don't need to be a data scientist, but you need to know the basics.

This exercise uses the [Titanic dataset](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv), a classic that's well-documented and easy to reason about.

**What you'll practice:**
- Loading a CSV from a URL
- Inspecting, filtering, and grouping data
- Exporting results to a new CSV

In [ ]:
!pip install pandas -q

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

## Inspect the data

Before doing anything, look at what you have. How many rows? Any missing values? What does the data actually mean?

In [ ]:
# Basic stats
print(df.dtypes)
print()
print("Missing values:")
print(df.isnull().sum())

## Filter

Select only the rows that match a condition. This is the most common thing you'll do when building RAG pipelines: filter documents by source, date, category, etc.

In [ ]:
# Passengers who survived and were in first class
survivors_first_class = df[(df["Survived"] == 1) & (df["Pclass"] == 1)]
print(f"First class survivors: {len(survivors_first_class)}")
survivors_first_class[["Name", "Age", "Sex", "Fare"]].head(10)

## Group by

Groupby lets you aggregate data by a category. Useful when you want to know things like: how many documents per source? What's the average chunk length by topic?

In [ ]:
# Survival rate by passenger class
survival_by_class = df.groupby("Pclass")["Survived"].agg(["sum", "count", "mean"])
survival_by_class.columns = ["survived", "total", "survival_rate"]
survival_by_class["survival_rate"] = survival_by_class["survival_rate"].round(2)
print(survival_by_class)

In [ ]:
# Survival rate by sex and class combined
df.groupby(["Sex", "Pclass"])["Survived"].mean().round(2)

## Export

Save the filtered result to a new CSV. In a real pipeline you'd do this to create a processed dataset before chunking and embedding.

In [ ]:
# Keep only the columns we care about, drop rows with missing age
clean = df[["Name", "Age", "Sex", "Pclass", "Survived"]].dropna()

clean.to_csv("titanic_clean.csv", index=False)
print(f"Saved {len(clean)} rows to titanic_clean.csv")

# Verify it loads back correctly
check = pd.read_csv("titanic_clean.csv")
check.head(3)

## Check your understanding

- What is the difference between `df["Age"]` and `df[["Age"]]`? Try both and compare the types.
- How would you find all passengers older than 60 who did not survive?
- What does `dropna()` do by default? How would you drop rows only where the `Age` column is missing?